In [26]:
import mlflow
mlflow.set_tracking_uri('http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/')

In [27]:
mlflow.set_experiment("Exp 5 Xgboost undersampling")

2026/03/16 12:27:25 INFO mlflow.tracking.fluent: Experiment with name 'Exp 5 Xgboost undersampling' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-bucket-1807077/25', creation_time=1773642446012, experiment_id='25', last_update_time=1773642446012, lifecycle_stage='active', name='Exp 5 Xgboost undersampling', tags={}>

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [29]:
df=pd.read_csv("processed_data.csv")
df['clean_comment'] = df['clean_comment'].fillna('')
df = df[['clean_comment', 'category']]
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [30]:
import platform
import optuna
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
# from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

In [31]:
from imblearn.under_sampling import RandomUnderSampler

# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# Bigram setting
ngram_range = (1, 2)
max_features = 1000

# Step 3: Train-test split before vectorization and resampling
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42
)

# Step 4: Vectorization using TF-IDF
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Step 5: Apply Random Under Sampling
rus = RandomUnderSampler(random_state=42)
X_train_vec, y_train = rus.fit_resample(X_train_vec, y_train)

/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


In [33]:
# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):

    with mlflow.start_run():

        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_Undersampling_TFIDF_Bigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)

        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log model
        mlflow.sklearn.log_model(model, f"{model_name}_model")


In [34]:
# Step 6: Optuna objective function for XGBoost
def objective_xgboost(trial):

    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int("max_depth", 3, 10)

    model = XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        random_state=42
    )

    return accuracy_score(
        y_test,
        model.fit(X_train_vec, y_train).predict(X_test_vec)
    )


In [35]:
# Step 7: Run Optuna for XGBoost
def run_optuna_experiment():

    study = optuna.create_study(direction="maximize")
    study.optimize(objective_xgboost, n_trials=30)

    # Get best parameters
    best_params = study.best_params

    best_model = XGBClassifier(
        n_estimators=best_params['n_estimators'],
        learning_rate=best_params['learning_rate'],
        max_depth=best_params['max_depth'],
        random_state=42
    )

    # Log best model with MLflow
    log_mlflow(
        "XGBoost",
        best_model,
        X_train_vec,
        X_test_vec,
        y_train,
        y_test
    )


# Run experiment
run_optuna_experiment()

[I 2026-03-16 12:28:35,525] A new study created in memory with name: no-name-197eec4c-a6f1-490d-bcf1-4a5e34d74bab
[I 2026-03-16 12:28:46,600] Trial 0 finished with value: 0.5604022285636635 and parameters: {'n_estimators': 178, 'learning_rate': 0.00013614753033057043, 'max_depth': 5}. Best is trial 0 with value: 0.5604022285636635.
[I 2026-03-16 12:28:57,302] Trial 1 finished with value: 0.7468406033428455 and parameters: {'n_estimators': 133, 'learning_rate': 0.06982252961841712, 'max_depth': 7}. Best is trial 1 with value: 0.7468406033428455.
[I 2026-03-16 12:29:09,088] Trial 2 finished with value: 0.7270009512161979 and parameters: {'n_estimators': 259, 'learning_rate': 0.03621998309218937, 'max_depth': 5}. Best is trial 1 with value: 0.7468406033428455.
[I 2026-03-16 12:29:22,530] Trial 3 finished with value: 0.7748335371653757 and parameters: {'n_estimators': 202, 'learning_rate': 0.09810892419197378, 'max_depth': 7}. Best is trial 3 with value: 0.7748335371653757.
[I 2026-03-16 1

🏃 View run XGBoost_Undersampling_TFIDF_Bigrams at: http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/#/experiments/25/runs/fda99c872292437a9e754b78dc9ef2a8
🧪 View experiment at: http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/#/experiments/25
